# 04 · Risk Assessment

Six statistical techniques applied to the district-year composite risk data from Notebook 01:

1. **CUSUM / EWMA** drift-detection charts for a couple of individual districts
2. **Outlier detection** — Z-score / modified Z-score (MAD), IQR fences, and multivariate
   Isolation Forest across the quality-metric feature space
3. **Benford's Law** goodness-of-fit check on enrollment counts (a classic screen for
   possible data fabrication or systemic entry error)
4. **Predictive risk model** — logistic regression and random forest classifying districts at
   risk of a "Needs Intervention" (or worse) determination, with ROC/AUC and feature importance
5. **Empirical Bayes shrinkage** for small-district indicator rates (stabilizing noisy small-n
   estimates by borrowing strength across similar districts)
6. A recap of the **composite risk index** construction from Notebook 01

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.subplots import make_subplots
from pathlib import Path
from scipy import stats
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
from sklearn.preprocessing import StandardScaler

DATA_DIR = Path("../data")
DOCS_DIR = Path("../docs")
PLOT_TEMPLATE = "plotly_white"

districts = pd.read_csv(DATA_DIR / "districts.csv")
submissions = pd.read_csv(DATA_DIR / "submissions.csv")
risk_scores = pd.read_csv(DATA_DIR / "risk_scores.csv")
idea_indicators = pd.read_csv(DATA_DIR / "idea_indicators.csv")
risk_scores = risk_scores.merge(
    districts[["district_id", "enrollment_total", "swd_pct", "urbanicity"]], on="district_id", how="left"
)
print(f"{len(risk_scores)} district-year risk records loaded.")
risk_scores.risk_tier.value_counts()

120 district-year risk records loaded.


risk_tier
Meets Requirements                72
Needs Assistance                  30
Needs Intervention                12
Needs Substantial Intervention     6
Name: count, dtype: int64

## 1. CUSUM & EWMA drift detection

A p-chart (Notebook 02) flags large single-month spikes but is slow to notice a *gradual* drift. CUSUM and EWMA are the standard complements for catching a slow deterioration before it crosses a hard control limit.

In [2]:
# Pick one drifting-looking district (highest late-arriving trend) and one stable district for contrast
monthly_district = submissions.groupby(["district_id", "period"]).validity_error_rate.mean().reset_index()
trend_slope = monthly_district.groupby("district_id").apply(
    lambda g: np.polyfit(range(len(g)), g.validity_error_rate.values, 1)[0], include_groups=False
).sort_values()
drifting_id = trend_slope.index[-1]
stable_id = trend_slope.index[len(trend_slope)//2]

def cusum_ewma(series, target=None, k=0.5, lam=0.2):
    s = series.values
    mu = target if target is not None else s.mean()
    sigma = s.std(ddof=0)
    # CUSUM (two-sided)
    c_pos, c_neg = [0], [0]
    for x in s:
        c_pos.append(max(0, c_pos[-1] + (x - mu) - k * sigma))
        c_neg.append(min(0, c_neg[-1] + (x - mu) + k * sigma))
    # EWMA
    ewma = [mu]
    for x in s:
        ewma.append(lam * x + (1 - lam) * ewma[-1])
    return np.array(c_pos[1:]), np.array(c_neg[1:]), np.array(ewma[1:])

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                     subplot_titles=(f"CUSUM — {districts.set_index('district_id').loc[drifting_id, 'district_name']} (drift example)",
                                      f"EWMA — same district, λ=0.2"))
sub = monthly_district[monthly_district.district_id == drifting_id].reset_index(drop=True)
cpos, cneg, ewma = cusum_ewma(sub.validity_error_rate)
fig.add_trace(go.Scatter(x=sub.period, y=cpos, name="CUSUM+", line=dict(color="#D64550")), row=1, col=1)
fig.add_trace(go.Scatter(x=sub.period, y=cneg, name="CUSUM-", line=dict(color="#12A594")), row=1, col=1)
fig.add_trace(go.Scatter(x=sub.period, y=sub.validity_error_rate, name="Raw error rate",
                          line=dict(color="#9AA6C3", dash="dot")), row=2, col=1)
fig.add_trace(go.Scatter(x=sub.period, y=ewma, name="EWMA", line=dict(color="#16213E", width=2)), row=2, col=1)
fig.update_layout(template=PLOT_TEMPLATE, height=560, title=None)
fig.show()

print(f"Steepest upward validity-error trend: {drifting_id} "
      f"({districts.set_index('district_id').loc[drifting_id, 'district_name']}), slope={trend_slope.iloc[-1]:.5f}/month")

Steepest upward validity-error trend: D100054 (Hillton CSD), slope=0.00071/month


## 2. Outlier detection

In [3]:
feat_cols = ["mean_completeness", "mean_validity_error", "on_time_rate", "mean_duplicate_rate", "pct_indicators_met"]
X = risk_scores[feat_cols].fillna(risk_scores[feat_cols].median())

# --- Univariate: modified Z-score (MAD-based) on validity error rate ---
med = X.mean_validity_error.median()
mad = stats.median_abs_deviation(X.mean_validity_error)
risk_scores["mod_zscore_error"] = 0.6745 * (X.mean_validity_error - med) / mad
uni_outliers = risk_scores[risk_scores.mod_zscore_error.abs() > 3.5]

# --- IQR fences on completeness ---
q1, q3 = X.mean_completeness.quantile([0.25, 0.75])
iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
iqr_outliers = risk_scores[(X.mean_completeness < lo) | (X.mean_completeness > hi)]

# --- Multivariate: Isolation Forest across all 5 quality features ---
iso = IsolationForest(n_estimators=300, contamination=0.08, random_state=207)
scaler = StandardScaler()
Xs = scaler.fit_transform(X)
risk_scores["iso_anomaly"] = iso.fit_predict(Xs) == -1
risk_scores["iso_score"] = -iso.score_samples(Xs)  # higher = more anomalous

print(f"Modified Z-score outliers (|z|>3.5) on validity error: {len(uni_outliers)}")
print(f"IQR-fence outliers on completeness: {len(iqr_outliers)}")
print(f"Isolation Forest multivariate anomalies: {risk_scores.iso_anomaly.sum()}")

Modified Z-score outliers (|z|>3.5) on validity error: 16
IQR-fence outliers on completeness: 16
Isolation Forest multivariate anomalies: 10


In [4]:
fig = px.scatter(risk_scores, x="mean_completeness", y="mean_validity_error",
                  color="iso_anomaly", size="iso_score", hover_data=["district_name", "region", "risk_tier"],
                  color_discrete_map={True: "#D64550", False: "#9AA6C3"},
                  template=PLOT_TEMPLATE, title="Isolation Forest — Multivariate Anomaly Detection",
                  labels={"mean_completeness": "Mean completeness", "mean_validity_error": "Mean validity error rate",
                          "iso_anomaly": "Flagged anomaly"})
fig.update_xaxes(tickformat=".0%")
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=440)
fig.show()

## 3. Benford's Law check on enrollment counts

Naturally occurring count data that arises from an unconstrained multiplicative growth process (organic population counts, transaction amounts, etc.) tends to follow Benford's leading-digit distribution. A significant deviation is a classic *screen* for possible fabricated or systematically altered figures — but it is not proof on its own, and it can also arise for entirely benign structural reasons, which is exactly what we'll see below.

In [5]:
def benford_expected():
    return np.array([np.log10(1 + 1/d) for d in range(1, 10)])

def leading_digit(n):
    n = abs(int(n))
    while n >= 10:
        n //= 10
    return n

leading_digits = districts.enrollment_total.apply(leading_digit)
observed_counts = leading_digits.value_counts().reindex(range(1, 10), fill_value=0).sort_index()
observed_pct = observed_counts / observed_counts.sum()
expected_pct = benford_expected()
expected_counts = expected_pct * observed_counts.sum()

chi2, p_value = stats.chisquare(observed_counts, expected_counts)

fig = go.Figure()
fig.add_trace(go.Bar(x=list(range(1,10)), y=observed_pct, name="Observed", marker_color="#12A594"))
fig.add_trace(go.Scatter(x=list(range(1,10)), y=expected_pct, name="Benford expected",
                          mode="lines+markers", line=dict(color="#D64550", width=2)))
fig.update_layout(template=PLOT_TEMPLATE, height=380, yaxis_tickformat=".0%",
                   title=f"Benford's Law — Leading Digit of District Enrollment (χ²={chi2:.2f}, p={p_value:.3f})",
                   xaxis_title="Leading digit", yaxis_title="Share of districts",
                   xaxis=dict(tickmode="array", tickvals=list(range(1,10))))
fig.show()

print(f"Chi-square goodness-of-fit vs. Benford distribution: χ²={chi2:.2f}, p={p_value:.3f}")
print("No significant deviation from Benford's Law." if p_value > 0.05 else
      "Statistically significant deviation from Benford's Law.")

Chi-square goodness-of-fit vs. Benford distribution: χ²=27.13, p=0.001
Statistically significant deviation from Benford's Law.


**Reading this result correctly matters.** The deviation here is *not* evidence of fabricated enrollment figures — it's an artifact of how districts were generated: three discrete size tiers (Large City / Suburban / Small-Rural), each spanning less than a full order of magnitude, mixed together in fixed proportions. That administratively stratified structure breaks the scale-invariance that Benford's Law assumes, independent of any data quality issue. In practice this is the single most common false-positive pattern for a Benford screen: **always check whether the population is a stratified mix of sub-groups before treating a deviation as a red flag.** The correct next step is not to conclude manipulation, but to re-run the test *within* each stratum (size tier) — where the assumption of a single multiplicative process is more plausible — before drawing any conclusion.

## 4. Predictive risk model

Classify each district-year as **elevated risk** (`Needs Intervention` or `Needs Substantial Intervention`) vs. **standard** (`Meets Requirements` / `Needs Assistance`). To keep this an honest predictive exercise, the model uses only **structural district characteristics known in advance** (enrollment, % students with disabilities, urbanicity, size tier, region) — *not* the quality metrics that directly define the risk tier, which would make the classification circular. This mirrors a real early-warning use case: flag likely-elevated-risk districts for closer monitoring using only what's known before the collection cycle even starts.

In [6]:
risk_scores["elevated_risk"] = risk_scores.risk_tier.isin(["Needs Intervention", "Needs Substantial Intervention"])

structural_num = ["enrollment_total", "swd_pct"]
structural_cat = ["urbanicity", "size_tier", "region"]
X_struct = pd.get_dummies(risk_scores[structural_num + structural_cat], columns=structural_cat, drop_first=True)
y = risk_scores.elevated_risk.astype(int)

X_train, X_test, y_train, y_test = train_test_split(X_struct, y, test_size=0.3, random_state=207, stratify=y)

logit = LogisticRegression(max_iter=1000, class_weight="balanced")
logit.fit(X_train, y_train)
logit_proba = logit.predict_proba(X_test)[:, 1]

rf = RandomForestClassifier(n_estimators=400, max_depth=4, random_state=207, class_weight="balanced")
rf.fit(X_train, y_train)
rf_proba = rf.predict_proba(X_test)[:, 1]

auc_logit = roc_auc_score(y_test, logit_proba)
auc_rf = roc_auc_score(y_test, rf_proba)
print(f"Logistic regression AUC: {auc_logit:.3f}")
print(f"Random forest AUC:       {auc_rf:.3f}")

fpr_l, tpr_l, _ = roc_curve(y_test, logit_proba)
fpr_r, tpr_r, _ = roc_curve(y_test, rf_proba)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr_l, y=tpr_l, name=f"Logistic (AUC={auc_logit:.2f})", line=dict(color="#16213E")))
fig.add_trace(go.Scatter(x=fpr_r, y=tpr_r, name=f"Random Forest (AUC={auc_rf:.2f})", line=dict(color="#12A594")))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], name="Chance", line=dict(color="#9AA6C3", dash="dot")))
fig.update_layout(template=PLOT_TEMPLATE, height=420, title="ROC Curves — Elevated-Risk Classification (structural features only)",
                   xaxis_title="False positive rate", yaxis_title="True positive rate")
fig.show()

Logistic regression AUC: 0.806
Random forest AUC:       0.852


In [7]:
importances = pd.Series(rf.feature_importances_, index=X_struct.columns).sort_values()
coefs = pd.Series(logit.coef_[0], index=X_struct.columns).sort_values()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Random Forest — Feature Importance", "Logistic Regression — Coefficients"))
fig.add_trace(go.Bar(y=importances.index, x=importances.values, orientation="h", marker_color="#12A594"), row=1, col=1)
fig.add_trace(go.Bar(y=coefs.index, x=coefs.values, orientation="h",
                      marker_color=np.where(coefs.values > 0, "#D64550", "#16213E")), row=1, col=2)
fig.update_layout(template=PLOT_TEMPLATE, height=420, showlegend=False,
                   title="What structural characteristics are associated with elevated risk?")
fig.show()

## 5. Empirical Bayes shrinkage for small-district indicator rates

Small districts have small denominators, so their raw compliance rates are noisy — a district with 8 eligible students can swing from 100% to 75% on one case. Empirical Bayes shrinkage pulls each district's estimate toward the regional mean, weighted by how much data it has (this is the same logic behind small-area disease-mapping estimates in spatial epidemiology).

In [8]:
ind = idea_indicators[idea_indicators.indicator_id == "Ind-11"].copy()  # timely initial evaluation
grand_mean = (ind.actual_rate * ind.denominator_n).sum() / ind.denominator_n.sum()

# Method-of-moments empirical Bayes (Beta-Binomial): estimate prior variance from between-district spread
weighted_var = np.average((ind.actual_rate - grand_mean) ** 2, weights=ind.denominator_n)
sampling_var = (grand_mean * (1 - grand_mean) / ind.denominator_n).mean()
tau2 = max(weighted_var - sampling_var, 1e-6)   # between-district variance

ind["shrinkage_weight"] = tau2 / (tau2 + (grand_mean * (1 - grand_mean) / ind.denominator_n))
ind["shrunk_rate"] = ind.shrinkage_weight * ind.actual_rate + (1 - ind.shrinkage_weight) * grand_mean

fig = px.scatter(ind, x="denominator_n", y="actual_rate", color="shrinkage_weight",
                  hover_data=["district_id"], template=PLOT_TEMPLATE,
                  color_continuous_scale=["#D64550", "#E8A33D", "#12A594"],
                  title="Funnel Plot — Raw Indicator Rate vs. Denominator Size (Ind-11: Timely Evaluation)",
                  labels={"denominator_n": "Denominator (n eligible)", "actual_rate": "Raw rate",
                          "shrinkage_weight": "Shrinkage weight (1=trust raw rate)"})
fig.add_hline(y=grand_mean, line_dash="dot", line_color="#16213E", annotation_text="Grand mean")
fig.update_yaxes(tickformat=".0%")
fig.update_layout(height=440)
fig.show()

compare = ind.nsmallest(8, "denominator_n")[["district_id", "denominator_n", "actual_rate", "shrunk_rate"]]
compare.columns = ["district_id", "n", "raw_rate", "shrunk_rate"]
print("Smallest-denominator districts: raw vs. shrunk estimate")
compare.round(3)

Smallest-denominator districts: raw vs. shrunk estimate


,district_id,n,raw_rate,shrunk_rate
118,D100010,5,0.978,0.960
328,D100028,5,0.971,0.959
94,D100008,7,0.960,0.956
502,D100042,7,0.971,0.960
610,D100051,7,0.951,0.954
214,D100018,8,0.958,0.956
244,D100021,9,0.967,0.959
364,D100031,10,0.998,0.972


## 6. Composite risk index recap

In [9]:
fig = px.histogram(risk_scores, x="risk_index", color="risk_tier", nbins=30, template=PLOT_TEMPLATE,
                    color_discrete_map={"Meets Requirements": "#12A594", "Needs Assistance": "#E8A33D",
                                         "Needs Intervention": "#D64550", "Needs Substantial Intervention": "#7A1F27"},
                    title="Composite Risk Index Distribution & Tiers", category_orders={
                        "risk_tier": ["Meets Requirements","Needs Assistance","Needs Intervention","Needs Substantial Intervention"]})
fig.update_layout(height=400, bargap=0.05)
fig.show()

tier_counts = risk_scores.risk_tier.value_counts()
tier_counts

risk_tier
Meets Requirements                72
Needs Assistance                  30
Needs Intervention                12
Needs Substantial Intervention     6
Name: count, dtype: int64

## 7. Export static dashboard page

In [10]:
import json
NAV_PAGES = json.load(open("../config/nav_pages.json"))

def render_nav(active):
    links = "\n".join(f'<a href="{href}" class="{"active" if key == active else ""}">{label}</a>'
                       for key, label, href in NAV_PAGES)
    return f'''<nav class="site-nav"><div class="nav-inner">
      <div class="brand"><span class="dot"></span> NYSED Data Integrity Platform</div>
      <div class="links">{links}</div></div></nav>'''

def page_shell(title, eyebrow, description, body_html, active, filename):
    html = f'''<!DOCTYPE html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>{title} · NYSED Data Integrity Platform</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Lora:wght@500;600;700&family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600;700&display=swap" rel="stylesheet">
<link rel="stylesheet" href="../assets/style.css">
</head><body>
{render_nav(active)}
<header class="dash-header"><div class="dash-header-inner">
  <div class="eyebrow">{eyebrow}</div><h1>{title}</h1><p>{description}</p>
</div></header>
<main class="dash-body">{body_html}</main>
<footer class="site-footer">Synthetic demonstration data · NYSED Data Integrity &amp; Risk Monitoring Platform ·
  <a href="https://github.com/zia207/NYSED-Data-Integrity" target="_blank">Source code</a></footer>
</body></html>'''
    out = DOCS_DIR / "dashboard" / filename
    out.write_text(html)
    print("wrote", out)

def kpi_card(label, value, delta=None, tone="good", warn=False):
    cls = "kpi-card" + (" warn" if warn else "")
    delta_html = f'<div class="kpi-delta {tone}">{delta}</div>' if delta else ""
    return f'<div class="{cls}"><div class="kpi-label">{label}</div><div class="kpi-value">{value}</div>{delta_html}</div>'

def fig_div(f, div_id, include_js=False):
    return pio.to_html(f, include_plotlyjs="inline" if include_js else False, full_html=False, div_id=div_id)

In [11]:
n_elevated = int(risk_scores.elevated_risk.sum())
kpi_html = f'''<div class="kpi-row">
  {kpi_card("Elevated-Risk District-Years", f"{n_elevated}", f"of {len(risk_scores)} total", warn=True, tone="bad")}
  {kpi_card("Isolation Forest Anomalies", f"{int(risk_scores.iso_anomaly.sum())}")}
  {kpi_card("Risk Model AUC (Random Forest)", f"{auc_rf:.2f}")}
  {kpi_card("Benford χ² p-value", f"{p_value:.3f}", "Deviation traced to size-tier stratification, not fabrication", tone="good")}
</div>'''

fig_tier_hist = px.histogram(risk_scores, x="risk_index", color="risk_tier", nbins=30, template=PLOT_TEMPLATE,
                    color_discrete_map={"Meets Requirements": "#12A594", "Needs Assistance": "#E8A33D",
                                         "Needs Intervention": "#D64550", "Needs Substantial Intervention": "#7A1F27"},
                    category_orders={"risk_tier": ["Meets Requirements","Needs Assistance","Needs Intervention","Needs Substantial Intervention"]})
fig_tier_hist.update_layout(height=360, bargap=0.05, margin=dict(t=20,l=10,r=10,b=10))

fig_iso_small = px.scatter(risk_scores, x="mean_completeness", y="mean_validity_error",
                  color="iso_anomaly", hover_data=["district_name"],
                  color_discrete_map={True: "#D64550", False: "#9AA6C3"}, template=PLOT_TEMPLATE)
fig_iso_small.update_xaxes(tickformat=".0%"); fig_iso_small.update_yaxes(tickformat=".0%")
fig_iso_small.update_layout(height=360, margin=dict(t=20,l=10,r=10,b=10))

fig_roc_small = go.Figure()
fig_roc_small.add_trace(go.Scatter(x=fpr_l, y=tpr_l, name=f"Logistic ({auc_logit:.2f})", line=dict(color="#16213E")))
fig_roc_small.add_trace(go.Scatter(x=fpr_r, y=tpr_r, name=f"Random Forest ({auc_rf:.2f})", line=dict(color="#12A594")))
fig_roc_small.add_trace(go.Scatter(x=[0,1], y=[0,1], line=dict(color="#9AA6C3", dash="dot"), showlegend=False))
fig_roc_small.update_layout(template=PLOT_TEMPLATE, height=360, margin=dict(t=20,l=10,r=10,b=10))

fig_benford_small = go.Figure()
fig_benford_small.add_trace(go.Bar(x=list(range(1,10)), y=observed_pct, marker_color="#12A594", name="Observed"))
fig_benford_small.add_trace(go.Scatter(x=list(range(1,10)), y=expected_pct, line=dict(color="#D64550", width=2), name="Benford"))
fig_benford_small.update_layout(template=PLOT_TEMPLATE, height=360, yaxis_tickformat=".0%", margin=dict(t=20,l=10,r=10,b=10))

body = kpi_html + f'''
<div class="chart-row">
  <div class="chart-panel"><h3>Composite risk index</h3>
    <div class="chart-note">{tier_counts.get("Meets Requirements",0)} Meets Requirements · {tier_counts.get("Needs Assistance",0)} Needs Assistance · {tier_counts.get("Needs Intervention",0)} Needs Intervention · {tier_counts.get("Needs Substantial Intervention",0)} Needs Substantial Intervention</div>
    {fig_div(fig_tier_hist, "fig-tier", include_js=True)}</div>
  <div class="chart-panel"><h3>Multivariate anomaly detection</h3>
    <div class="chart-note">Isolation Forest across 5 quality features; {int(risk_scores.iso_anomaly.sum())} flagged.</div>
    {fig_div(fig_iso_small, "fig-iso")}</div>
</div>
<div class="chart-row">
  <div class="chart-panel"><h3>Elevated-risk prediction — ROC</h3>
    <div class="chart-note">Random forest AUC {auc_rf:.2f} vs. logistic regression AUC {auc_logit:.2f}.</div>
    {fig_div(fig_roc_small, "fig-roc")}</div>
  <div class="chart-panel"><h3>Benford's Law — enrollment leading digit</h3>
    <div class="chart-note">χ²={chi2:.2f}, p={p_value:.3f} — deviation reflects stratified size tiers, not a data-quality flag (see notebook for the diagnostic reasoning).</div>
    {fig_div(fig_benford_small, "fig-benford")}</div>
</div>
<div class="chart-panel"><h3>Districts flagged by Isolation Forest</h3>
  <table class="kpi-table"><thead><tr><th>District</th><th>Region</th><th>Risk Tier</th><th>Completeness</th><th>Validity Error</th></tr></thead><tbody>
  {"".join(f"<tr><td>{r.district_name}</td><td>{r.region}</td><td><span class='pill pill-risk'>{r.risk_tier}</span></td><td class='target'>{r.mean_completeness:.1%}</td><td class='target'>{r.mean_validity_error:.1%}</td></tr>" for r in risk_scores[risk_scores.iso_anomaly].itertuples())}
  </tbody></table>
</div>
'''

page_shell(
    title="Risk Assessment",
    eyebrow="Notebook 04",
    description="Control-chart drift detection, univariate &amp; multivariate outlier detection, Benford's Law, a predictive risk model (logistic regression / random forest), and empirical Bayes shrinkage for small districts.",
    body_html=body, active="risk-assessment.html", filename="risk-assessment.html",
)

wrote ../docs/dashboard/risk-assessment.html
